# Exercises (Student) - MCP Client with LLM

In [1]:
!pip install -q mcp nest_asyncio requests

In [2]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [3]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [7]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

#To-Do: Add decorator to register add as a tool
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# TODO (optional exercise): add multiply(a, b) here if you want an extra tool
@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

#To-Do: Add decorator to register greet as a tool
@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting string."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()


Overwriting server.py


## Exercise 1 (provide answer)

#To-Do: Why is STDIO transport simple for local MCP dev compared to HTTP?
STDIO transport is simpler for local MCP development because the client can start the server as a local process and communicate through standard input/output. Unlike HTTP, it does not require setting up a web server, choosing ports, handling URLs, CORS, authentication, or deployment. This makes it easier to test and debug MCP tools locally.

## Exercise 2

In [10]:

import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = params = StdioServerParameters(command="mcp", args=["run", "server.py"]) #To-Do: Create StdioServerParameters with command to run server.py
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()


In [11]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [15]:
async def ex3_list():
    params = StdioServerParameters(command="mcp", args=["run", "server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))


In [16]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
multiply {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4

#To-Do: Explain how the conversion to llm tool happens in MCP server code ?

The conversion takes the MCP tool metadata and reshapes it into the LLM function-calling format. The MCP tool name becomes the function name, the MCP description becomes the function description, and the MCP inputSchema becomes the function parameters. This allows the LLM to understand which tools are available, what each tool does, and what arguments it needs to provide.

In [17]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [18]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [20]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="mcp", args=["run", "server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in tools.tools]
            calls = call_llm(prompt, functions)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])


In [ ]:
await ex5_run("Add 2 to 20")

tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun

In [22]:
def stub_plan(prompt, functions):
    text = prompt.lower()

    if "multiply" in text:
        return [{"name": "multiply", "args": {"a": 6, "b": 7}}]

    if "add" in text:
        return [{"name": "add", "args": {"a": 2, "b": 20}}]

    return []

In [23]:
await ex5_run("Multiply 6 by 7")

tool_calls: [{'name': 'multiply', 'args': {'a': 6, 'b': 7}}]
result: ['42']
